# Step 5: Export the Assam inputs from Earth Engine

**Run this in Colab** (it needs your Earth Engine login). Runtime: CPU is fine.

It exports four GeoTIFFs to your Google Drive, folder `cutoff_assam`, all on **exactly the same 10 m grid** (UTM zone 46N):

| File | Bands | Stored as | To get real values |
|---|---|---|---|
| `s1_pre_20260717.tif` | VV, VH | int16 | divide by 100 (dB) |
| `s1_post_20260729.tif` | VV, VH | int16 | divide by 100 (dB) |
| `aux_jrc_hand.tif` | JRC occurrence, HAND | int16 | JRC: % as is; HAND: divide by 10 (m) |

No-data value in all files: **-32768**.

Why int16 and not float: it halves the file size (about 0.6 GB per radar file instead of 1.1 GB), and 0.01 dB precision is far finer than radar noise.

**No smoothing** is applied to the radar, so the images match the training data (Sen1Floods11 also uses raw Earth Engine Sentinel-1 values in dB).

After the exports finish (about 10 to 40 minutes), download the folder from Drive. Step 6 runs the models on these files on Kaggle.

In [ ]:
!pip install -q earthengine-api

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.insert(0, REPO)
from src import config
print("Project code loaded")

In [ ]:
import ee
PROJECT_ID = "deep-learning-512620"
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready")

## 1. Find the two radar images

Looks up the image on our track (orbit 4, descending) for each date and checks it is the same image we chose in Check 2.

In [ ]:
aoi = ee.Geometry.Rectangle(list(config.STUDY_BBOX))

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(aoi)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.eq("relativeOrbitNumber_start", config.TRACK["orbit"]))
      .filter(ee.Filter.eq("orbitProperties_pass", config.TRACK["pass"]))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

def find(date, known_id):
    day = ee.Date(date)
    col = s1.filterDate(day, day.advance(1, "day"))
    ids = col.aggregate_array("system:index").getInfo()
    assert ids, f"No image found on {date} for our track"
    img_id = known_id if known_id in ids else ids[0]
    if img_id != known_id:
        print(f"WARNING: {date}: expected {known_id}, using {img_id}. Note this in NOTES.md.")
    return img_id

PRE_ID = find(config.PRE_DATE, config.PRE_ID_KNOWN)
POST_ID = find(config.POST_DATE, config.POST_ID_KNOWN)
print("Before (17 July):", PRE_ID)
print("During (29 July):", POST_ID)

## 2. Fix one grid for every file

All files use the same corner, pixel size and size, so pixel (row, col) is the same place in every file. This is what lets us compare the radar, the permanent water and the terrain pixel by pixel.

In [ ]:
import math
CRS = "EPSG:32646"   # UTM zone 46N, metres
PIX = 10

b = aoi.transform(CRS, 1).bounds(1, CRS).coordinates().getInfo()[0]
xs = [p[0] for p in b]; ys = [p[1] for p in b]
x0 = math.floor(min(xs) / PIX) * PIX
y1 = math.ceil(max(ys) / PIX) * PIX
x1 = math.ceil(max(xs) / PIX) * PIX
y0 = math.floor(min(ys) / PIX) * PIX
W, H = (x1 - x0) // PIX, (y1 - y0) // PIX

TRANSFORM = [PIX, 0, x0, 0, -PIX, y1]
REGION = ee.Geometry.Rectangle([x0, y0, x1, y1], CRS, False)
SHARD = 256
FILE_DIM = int(math.ceil(max(W, H) / SHARD) * SHARD)   # big enough for one file per export

print(f"Grid: {W} x {H} pixels ({W*PIX/1000:.0f} km x {H*PIX/1000:.0f} km)")
print("Transform:", TRANSFORM)

## 3. Build the images

In [ ]:
NODATA = -32768

def radar(img_id):
    img = ee.Image(f"COPERNICUS/S1_GRD/{img_id}").select(["VV", "VH"])
    return img.multiply(100).round().toInt16().unmask(NODATA)

pre = radar(PRE_ID)
post = radar(POST_ID)

# Permanent water: % of time water was seen, 1984 onward. Never-water areas are
# masked in the dataset, so they become 0 (not no-data).
jrc = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
       .unmask(0).toInt16().rename("jrc_occurrence"))

# Height above nearest drainage (m), about 90 m pixels: resample smoothly to 10 m.
hand = (ee.Image("MERIT/Hydro/v1_0_1").select("hnd")
        .resample("bilinear").multiply(10).round().toInt16()
        .unmask(NODATA).rename("hand_dm"))

aux = jrc.addBands(hand)

## 4. Quick look before exporting

Both radar images should show the study area, with the rivers dark. The terrain image should be dark (low) along the Brahmaputra plain and bright in the hills to the south-east.

In [ ]:
from IPython.display import Image, display
th = {"region": aoi, "dimensions": 600}
print("Before, VV"); display(Image(url=pre.select("VV").divide(100).getThumbURL({**th, "min": -25, "max": 0})))
print("During, VV"); display(Image(url=post.select("VV").divide(100).getThumbURL({**th, "min": -25, "max": 0})))
print("HAND (m)");   display(Image(url=hand.divide(10).getThumbURL({**th, "min": 0, "max": 50})))

## 5. Start the exports

In [ ]:
FOLDER = "cutoff_assam"
jobs = {
    "s1_pre_20260717": pre,
    "s1_post_20260729": post,
    "aux_jrc_hand": aux,
}
tasks = []
for name, img in jobs.items():
    t = ee.batch.Export.image.toDrive(
        image=img, description=name, folder=FOLDER, fileNamePrefix=name,
        crs=CRS, crsTransform=TRANSFORM, region=REGION,
        maxPixels=1e10, fileFormat="GeoTIFF", fileDimensions=FILE_DIM,
        formatOptions={"cloudOptimized": True, "noData": NODATA},
    )
    t.start()
    tasks.append((name, t))
    print("Started:", name)
print("\nYou can close this tab now; the exports run on Google's servers.")
print("Track them at https://code.earthengine.google.com/tasks or run the next cell.")

In [ ]:
# Check progress (rerun this cell, or leave it running).
import time
while True:
    states = {n: t.status()["state"] for n, t in tasks}
    print(time.strftime("%H:%M:%S"), states)
    if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states.values()):
        break
    time.sleep(60)
for n, t in tasks:
    s = t.status()
    if s["state"] != "COMPLETED":
        print(n, "->", s.get("error_message"))

## 6. Save a record of what was exported

In [ ]:
import json, datetime
info = {
    "exported_on": datetime.date.today().isoformat(),
    "pre_id": PRE_ID, "post_id": POST_ID,
    "crs": CRS, "transform": TRANSFORM, "width": W, "height": H,
    "nodata": NODATA,
    "scaling": {"VV": "dB x 100", "VH": "dB x 100",
                "jrc_occurrence": "percent", "hand_dm": "metres x 10"},
    "jrc": "JRC/GSW1_4/GlobalSurfaceWater occurrence",
    "hand": "MERIT/Hydro/v1_0_1 hnd (bilinear)",
    "files": [n + ".tif" for n in jobs],
}
os.makedirs(f"{REPO}/results", exist_ok=True)
with open(f"{REPO}/results/assam_export_info.json", "w") as f:
    json.dump(info, f, indent=1)
print(json.dumps(info, indent=1))
print("\nCopy this into NOTES.md, or download results/assam_export_info.json and commit it.")

## Next

1. In Google Drive, open the `cutoff_assam` folder and download the three `.tif` files. If Earth Engine split a file into parts (names ending in `-0000000000-0000000000`), download all parts.
2. On Kaggle: **Datasets, New Dataset**, name it `cutoff-assam-inputs`, upload the `.tif` files. Add the five `.pth` model files as a second dataset, `cutoff-models`.
3. Run Step 6 (`06_predict_assam.ipynb`) on Kaggle.

**Do not use "Save a copy in GitHub" from Colab** for this notebook (see README).